In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("GROQ_API_KEY configured:", bool(groq_api_key))

GROQ_API_KEY configured: True


In [2]:
from langchain_groq import ChatGroq

# Replace this model if your Groq dashboard shows a different available model.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

In [3]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from groq import Groq

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\ce\AppData\Local\Temp\ipykernel_21628\3779550650.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever
c:\Users\ce\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
doc_list = [

    """
    Python is a popular programming language used for web development,
    data science, machine learning, automation, and artificial intelligence.
    Python has a simple syntax and a large ecosystem of libraries.
    """,

    """
    Machine learning is a branch of artificial intelligence that allows
    computers to learn patterns from data without being explicitly programmed.
    Common machine learning algorithms include linear regression,
    decision trees, random forests, KNN, and support vector machines.
    """,

    """
    Retrieval Augmented Generation, commonly called RAG, combines information
    retrieval with large language models. A RAG system first retrieves relevant
    documents and then provides those documents as context to an LLM.
    """,

    """
    Hybrid search combines semantic vector search with traditional keyword
    search. Vector search understands the meaning of a query, while keyword
    search is useful for exact words, product codes, invoice numbers,
    technical terms, and identifiers.
    """,

    """
    BM25 is a lexical information retrieval algorithm. It ranks documents
    according to how frequently query terms occur in a document while also
    considering document length and term rarity.
    """,

    """
    Vector databases store numerical embeddings generated from text, images,
    or other data. Examples of vector databases include Chroma, Pinecone,
    Weaviate, Milvus, and FAISS.
    """,

    """
    Chroma is an open-source vector database designed for AI applications.
    It can store embeddings and metadata and perform similarity searches
    over documents.
    """,

    """
    Pinecone is a managed vector database designed for production AI
    applications. It provides vector search, metadata filtering, and
    infrastructure for scalable retrieval systems.
    """,

    """
    LangChain is a framework for building applications powered by large
    language models. It provides components for document loading,
    text splitting, embeddings, retrievers, prompts, and RAG pipelines.
    """,

    """
    Groq provides fast inference for large language models. A RAG application
    can retrieve relevant documents from a vector database and then send
    the retrieved context to a Groq-hosted language model to generate an answer.
    """,

    """
    Hugging Face provides machine learning models and libraries that can be
    used for natural language processing, embeddings, computer vision,
    and generative AI applications.
    """,

    """
    The invoice INV-2026-001245 belongs to ABC Technologies.
    The invoice contains PO number PO-4500012345, GSTIN 19ABCDE1234F1Z5,
    total amount of INR 125,450, and HSN code 847130.
    """,

    """
    The invoice INV-2026-001246 belongs to XYZ Industries.
    The invoice contains PO number PO-4500012346, GSTIN 27XYZDE5678G2Z9,
    total amount of INR 98,750, and HSN code 851712.
    """,

    """
    Invoice processing systems use OCR to extract information such as
    invoice number, invoice date, vendor name, GSTIN, purchase order,
    HSN code, quantity, unit price, tax amount, and total amount.
    """,

    """
    Semantic search is useful when a user asks a question using different
    words from the source document. For example, a query asking about
    programming languages may retrieve a document discussing Python even
    when the exact phrase is not present.
    """
]

In [6]:
# CONVERT THESE FOR DOCUMENTS OBJECT 

from langchain_core.documents import Document

documents = [
    Document(page_content=text)
    for text in doc_list
]

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 15

First chunk preview:

Python is a popular programming language used for web development,
    data science, machine learning, automation, and artificial intelligence.
    Python has a simple syntax and a large ecosystem of libraries.


In [8]:
from langchain_community.retrievers import BM25Retriever

# BM25 should use the SAME chunks that Chroma uses
bm25_retriever = BM25Retriever.from_documents(
    chunks
)

bm25_retriever.k = 4

print("BM25 retriever initialized")

BM25 retriever initialized


In [9]:
query = "Which invoice has GSTIN 19ABCDE1234F1Z5?"

results = bm25_retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
The invoice INV-2026-001246 belongs to XYZ Industries.
    The invoice contains PO number PO-4500012346, GSTIN 27XYZDE5678G2Z9,
    total amount of INR 98,750, and HSN code 851712.

--- Result 2 ---
The invoice INV-2026-001245 belongs to ABC Technologies.
    The invoice contains PO number PO-4500012345, GSTIN 19ABCDE1234F1Z5,
    total amount of INR 125,450, and HSN code 847130.

--- Result 3 ---
Python is a popular programming language used for web development,
    data science, machine learning, automation, and artificial intelligence.
    Python has a simple syntax and a large ecosystem of libraries.

--- Result 4 ---
Invoice processing systems use OCR to extract information such as
    invoice number, invoice date, vendor name, GSTIN, purchase order,
    HSN code, quantity, unit price, tax amount, and total amount.


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3114.28it/s]


Embedding dimensions: 384


In [12]:
from langchain_chroma import Chroma

PERSIST_DIR = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="test_demo",
)

print("Vectorstore created and persisted at:", PERSIST_DIR)

# Sanity check
query = "What is Agentic RAG?"
results = vectorstore.similarity_search(query, k=3)

print(f"\nTop {len(results)} results for query: '{query}'\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:300])
    print()

Vectorstore created and persisted at: ./chroma_db

Top 3 results for query: 'What is Agentic RAG?'

--- Result 1 ---
Page: None
Retrieval Augmented Generation, commonly called RAG, combines information
    retrieval with large language models. A RAG system first retrieves relevant
    documents and then provides those documents as context to an LLM.

--- Result 2 ---
Page: None
Retrieval Augmented Generation, commonly called RAG, combines information
    retrieval with large language models. A RAG system first retrieves relevant
    documents and then provides those documents as context to an LLM.

--- Result 3 ---
Page: None
Retrieval Augmented Generation, commonly called RAG, combines information
    retrieval with large language models. A RAG system first retrieves relevant
    documents and then provides those documents as context to an LLM.



In [13]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [14]:
from langchain_classic.retrievers import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        retriever
    ],
    weights=[
        0.5,  # BM25
        0.5   # Chroma
    ]
)

print("Hybrid retriever initialized")

Hybrid retriever initialized


In [15]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda
)




prompt = ChatPromptTemplate.from_template(
    """
Answer the question based ONLY on the provided context.

Do not use outside knowledge.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)



def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )




rag_chain = (
    RunnablePassthrough.assign(
        context=itemgetter("question") | ensemble_retriever
    )
    | RunnablePassthrough.assign(
        answer=(
            RunnableLambda(
                lambda x: {
                    "context": format_docs(x["context"]),
                    "question": x["question"]
                }
            )
            | prompt
            | llm
            | StrOutputParser()
        )
    )
)

In [17]:
query = "Which invoice has GSTIN 19ABCDE1234F1Z5??"

response = rag_chain.invoke({
    "question": query
})

print(response["answer"])

INV-2026-001245


In [18]:
print("QUESTION:")
print(response["question"])

print("\nRETRIEVED DOCUMENTS:")

for i, doc in enumerate(response["context"], 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)

print("\nANSWER:")
print(response["answer"])

QUESTION:
Which invoice has GSTIN 19ABCDE1234F1Z5??

RETRIEVED DOCUMENTS:

--- Document 1 ---
The invoice INV-2026-001245 belongs to ABC Technologies.
    The invoice contains PO number PO-4500012345, GSTIN 19ABCDE1234F1Z5,
    total amount of INR 125,450, and HSN code 847130.

--- Document 2 ---
The invoice INV-2026-001246 belongs to XYZ Industries.
    The invoice contains PO number PO-4500012346, GSTIN 27XYZDE5678G2Z9,
    total amount of INR 98,750, and HSN code 851712.

--- Document 3 ---
Python is a popular programming language used for web development,
    data science, machine learning, automation, and artificial intelligence.
    Python has a simple syntax and a large ecosystem of libraries.

--- Document 4 ---
Invoice processing systems use OCR to extract information such as
    invoice number, invoice date, vendor name, GSTIN, purchase order,
    HSN code, quantity, unit price, tax amount, and total amount.

ANSWER:
INV-2026-001245


# Test reranker independently

In [19]:
from sentence_transformers import CrossEncoder

In [21]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)

print("Reranker loaded!")

c:\Users\ce\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ce\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10744.13it/s]


Reranker loaded!


In [22]:
query = "Which invoice has GSTIN 19ABCDE1234F1Z5?"

retrieved_docs = ensemble_retriever.invoke(query)

pairs = [
    [query, doc.page_content]
    for doc in retrieved_docs
]

scores = reranker.predict(pairs)

ranked_docs = sorted(
    zip(scores, retrieved_docs),
    key=lambda x: x[0],
    reverse=True
)

for rank, (score, doc) in enumerate(ranked_docs, 1):
    print("=" * 70)
    print("Rank:", rank)
    print("Score:", score)
    print(doc.page_content)

Rank: 1
Score: 9.187822
The invoice INV-2026-001245 belongs to ABC Technologies.
    The invoice contains PO number PO-4500012345, GSTIN 19ABCDE1234F1Z5,
    total amount of INR 125,450, and HSN code 847130.
Rank: 2
Score: 5.5886736
The invoice INV-2026-001246 belongs to XYZ Industries.
    The invoice contains PO number PO-4500012346, GSTIN 27XYZDE5678G2Z9,
    total amount of INR 98,750, and HSN code 851712.
Rank: 3
Score: -2.7187014
Invoice processing systems use OCR to extract information such as
    invoice number, invoice date, vendor name, GSTIN, purchase order,
    HSN code, quantity, unit price, tax amount, and total amount.
Rank: 4
Score: -11.414579
Python is a popular programming language used for web development,
    data science, machine learning, automation, and artificial intelligence.
    Python has a simple syntax and a large ecosystem of libraries.
